## Model generation for Allen Visual dataset

This notebook generates 12 models as used in the [analysis](usage_demo.ipynb) and in the [report](riccardo_final_report.pdf). It is meant to be run on [Google Colab](https://colab.research.google.com/) for GPU usage.

If you have not downloaded the necessary dataset to train the models please refer to the top of the this [demo](UsageDemo.ipynb).

The notebook follows this structure:
1. Load Data
2. Train Models:
  - Single: 1 untrained + 5 Trained (mouse 4)
  - Multi: 1 untrained + 5 Trained (all mice)
3. Saves the models in Drive

### Install necessary packages and mount goolge drive

In [ ]:
!pip install --pre 'cebra[datasets,demos]'

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### Load data

In order to load data make sure you are in a directory where the data folder /data is located.

In [ ]:
import os
from cebra import CEBRA
import sys
import cebra_lens as lens

In [ ]:
#try with offset5 model and also offset15, offset36

In [ ]:
train_datas, valid_datas, discrete_labels_train, discrete_labels_val = (
    lens.utils_allen.get_single_session_datasets()
)

parameters = {
    "lr": 3e-4,
    "model_architechture": "offset5-model",
    "num_units": 32,
    "output_dim": 128,
    "temp": 1,
    "time_offsets": 10,
    "batch_size": 2042,
}


mice = ["mouse1", "mouse2", "mouse3", "mouse4"]

### Train Models

#### Single-session

In [ ]:
max_iterations = 10000

In [ ]:
# RUN MOUSE4 5 TIMES:
for i in range(5):

    cebra_model_single_session = CEBRA(
        model_architecture=parameters["model_architechture"],
        batch_size=parameters["batch_size"],
        learning_rate=parameters["lr"],
        temperature=parameters["temp"],
        num_hidden_units=parameters["num_units"],
        output_dimension=parameters["output_dim"],
        max_iterations=max_iterations,
        distance="cosine",
        conditional="time_delta",
        device="cuda",
        verbose=True,
        time_offsets=parameters["time_offsets"],
    )

    cebra_model_single_session.fit(train_datas[3].neural, train_datas[3].index)
    embeddings_single_session = cebra_model_single_session.transform(
        train_datas[3].neural
    )

    ################### SAVING ###################

    # torch version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_{i}_torch.pt",
    )
    cebra_model_single_session.save(model_path, backend="torch")
    print("Torch model saved under: ", model_path)
    # sklearn version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_{i}.pt",
    )
    cebra_model_single_session.save(model_path)
    print("Sklearn Model saved under: ", model_path)

In [ ]:
# UNTRAINED
max_iterations = 1

cebra_model_single_session = CEBRA(
    model_architecture=parameters["model_architechture"],
    batch_size=parameters["batch_size"],
    learning_rate=parameters["lr"],
    temperature=parameters["temp"],
    num_hidden_units=parameters["num_units"],
    output_dimension=parameters["output_dim"],
    max_iterations=max_iterations,
    distance="cosine",
    conditional="time_delta",
    device="cuda_if_available",
    verbose=True,
    time_offsets=parameters["time_offsets"],
)

cebra_model_single_session.fit(train_datas[3].neural, train_datas[3].index)
embeddings_single_session = cebra_model_single_session.transform(train_datas[3].neural)

################### SAVING ###################

# torch version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_UT_torch.pt",
)
cebra_model_single_session.save(model_path, backend="torch")
print("Torch model saved under: ", model_path)
# sklearn version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_single_session_{mice[3]}_{int(max_iterations/1000)}k_UT.pt",
)
cebra_model_single_session.save(model_path)
print("Sklearn Model saved under: ", model_path)

#### Multi-session

In [ ]:
untrained = False

multi_session_neural_train = []
multi_session_index_train = []
multi_session_neural_valid = []
multi_session_index_valid = []

for i in range(4):
    multi_session_neural_train.append(train_datas[i].neural)
    multi_session_index_train.append(train_datas[i].index)

    multi_session_neural_valid.append(valid_datas[i].neural)
    multi_session_index_valid.append(valid_datas[i].index)


max_iterations = 10000

In [ ]:
# TRAIN 5 MULTI-SESSIONS

for i in range(5):
    # Multisession training
    cebra_model_multi_session = CEBRA(
        model_architecture=parameters["model_architechture"],
        batch_size=parameters["batch_size"],
        learning_rate=parameters["lr"],
        temperature=parameters["temp"],
        num_hidden_units=parameters["num_units"],
        output_dimension=parameters["output_dim"],
        max_iterations=max_iterations,
        distance="cosine",
        conditional="time_delta",
        device="cuda_if_available",
        verbose=True,
        time_offsets=parameters["time_offsets"],
    )

    # Provide a list of data, i.e. datas = [data_a, data_b, ...]
    cebra_model_multi_session.fit(multi_session_neural_train, multi_session_index_train)

    ################### SAVING ###################

    # torch version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_{i}_torch.pt",
    )
    cebra_model_multi_session.save(model_path, backend="torch")
    print("Torch Model saved under: ", model_path)
    # sklearn version
    model_path = os.path.join(
        os.environ["DATA_PATH"],
        f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_{i}.pt",
    )
    cebra_model_multi_session.save(model_path)
    print("Sklearn Model saved under: ", model_path)

In [ ]:
# UNTRAINED

max_iterations = 1

# Multisession training
cebra_model_multi_session = CEBRA(
    model_architecture=parameters["model_architechture"],
    batch_size=parameters["batch_size"],
    learning_rate=parameters["lr"],
    temperature=parameters["temp"],
    num_hidden_units=parameters["num_units"],
    output_dimension=parameters["output_dim"],
    max_iterations=max_iterations,
    distance="cosine",
    conditional="time_delta",
    device="cuda_if_available",
    verbose=True,
    time_offsets=parameters["time_offsets"],
)

# Provide a list of data, i.e. datas = [data_a, data_b, ...]
cebra_model_multi_session.fit(multi_session_neural_train, multi_session_index_train)

################### SAVING ###################

# torch version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_UT_torch.pt",
)
cebra_model_multi_session.save(model_path, backend="torch")
print("Torch Model saved under: ", model_path)
# sklearn version
model_path = os.path.join(
    os.environ["DATA_PATH"],
    f"CEBRA_models/FinalModels/VISION/allen_multi_session_{int(max_iterations/1000)}k_UT.pt",
)
cebra_model_multi_session.save(model_path)
print("Sklearn Model saved under: ", model_path)